# Explore an annotation run

Reads a prediction JSONL written by `annotate_openrouter.py`, flattens the relations
into tables, checks the evidence quotes against the source text, and writes an Excel
workbook with one sheet per view.

Set `RUN_PATH` in the next cell to the run you want to inspect.

In [4]:
import json
import re
import unicodedata
from collections import Counter
from pathlib import Path

import pandas as pd

# Works whether the notebook is run from src/ or from the repository root.
REPO = Path.cwd().parent if Path.cwd().name == "src" else Path.cwd()

RUN_PATH = REPO / "runs" / "predictions_chr_deepseekv4pro0813_NotChunked.jsonl"
CORPUS_PATH = REPO / "data" / "corpus_chr.jsonl"
#XLSX_PATH = REPO / "runs" / "gide_opus5_relations.xlsx"

CLOSED_FIELDS = [
    "relation_type", "structure",
    "status", "reciprocity", "exclusivity", "adultery",
    "explicitness", "explicitness_in_plot",
    "centrality", "framing", "outcome",
    "confidence",
]
REL_COLUMNS = ["novel_id","orientation", "model", "relation_id", "characters"] + CLOSED_FIELDS + ["n_evidence", "comment"]
EV_COLUMNS = ["novel_id", "relation_id", "evidence_index", "quote", "locator", "description"]

pd.set_option("display.max_colwidth", 70)
pd.set_option("display.width", 220)

print("run    :", RUN_PATH, "| exists:", RUN_PATH.exists())
print("corpus :", CORPUS_PATH, "| exists:", CORPUS_PATH.exists())

run    : c:\Users\plathoon\Desktop\Aurelien\annotation_literature_relationships-main\runs\predictions_chr_deepseekv4pro0813_NotChunked.jsonl | exists: True
corpus : c:\Users\plathoon\Desktop\Aurelien\annotation_literature_relationships-main\data\corpus_chr.jsonl | exists: True


## 1. Load the run and check it succeeded

A failed request still produces a record, with the reason in `runtime_error`,
`parse_error`, or `validation_error`. Check this before reading the tables below.

In [5]:
records = [
    json.loads(line)
    for line in RUN_PATH.read_text(encoding="utf-8").splitlines()
    if line.strip()
]
print(f"{len(records)} record(s)\n")

n_failed = 0
for r in records:
    print(f"{r.get('novel_id')} | {r.get('model')} | ok={r.get('ok')}")
    for key in ("runtime_error", "parse_error", "validation_error"):
        if r.get(key):
            n_failed += 1
            print(f"    {key}: {str(r[key])[:800]}")
    usage = (r.get("raw_response") or {}).get("usage")
    if usage:
        print(f"    usage: {usage}")

if n_failed:
    print("\n>> This run has failures. The tables below will be empty until it is re-run.")

15 record(s)

adam_paienne | deepseek/deepseek-v4-pro | ok=True
    usage: {'prompt_tokens': 52114, 'completion_tokens': 4327, 'total_tokens': 56441, 'cost': 0.019589888, 'is_byok': False, 'prompt_tokens_details': {'cached_tokens': 51200, 'cache_write_tokens': 0, 'audio_tokens': 0, 'video_tokens': 0}, 'cost_details': {'upstream_inference_cost': 0.019589888, 'upstream_inference_prompt_cost': 0.007335824, 'upstream_inference_completions_cost': 0.012254064}, 'completion_tokens_details': {'reasoning_tokens': 3669, 'image_tokens': 0, 'audio_tokens': 0}}
alanic_roses | deepseek/deepseek-v4-pro | ok=True
    usage: {'prompt_tokens': 117688, 'completion_tokens': 3287, 'total_tokens': 120975, 'cost': 0.10810794, 'is_byok': False, 'prompt_tokens_details': {'cached_tokens': 0, 'cache_write_tokens': 0, 'audio_tokens': 0, 'video_tokens': 0}, 'cost_details': {'upstream_inference_cost': 0.10810794, 'upstream_inference_prompt_cost': 0.10238856, 'upstream_inference_completions_cost': 0.00571938}, 'comp

## 2. Relations table

One row per annotated relationship, with the eleven closed variables grouped as in the codebook.

In [ ]:
records[0].keys()

In [6]:
def relation_rows(records):
    for r in records:
        annotation = r.get("annotation") or {}
        for rel in annotation.get("relations") or []:
            row = {
                "novel_id": r.get("novel_id"),
                'orientation': r.get("orientation"),
                "model": r.get("model"),
                "relation_id": rel.get("relation_id"),
                "characters": " | ".join(str(c) for c in rel.get("characters") or []),
                "n_evidence": len(rel.get("evidence") or []),
                "comment": rel.get("comment"),
            }
            row.update({f: rel.get(f) for f in CLOSED_FIELDS})
            yield row


relations = pd.DataFrame(list(relation_rows(records)), columns=REL_COLUMNS)
print(f"{len(relations)} relation(s)")
relations

69 relation(s)


,novel_id,orientation,model,relation_id,characters,relation_type,structure,status,reciprocity,exclusivity,adultery,explicitness,explicitness_in_plot,centrality,framing,outcome,confidence,n_evidence,comment
0,adam_paienne,hetero,deepseek/deepseek-v4-pro,R1,Mélissandre de Noves | Tiburce Gardanne,female_male,dyadic,romantic_relationship,reciprocal,adultery,external_adultery,explicit,hidden_except,main,positive,stable_or_open,high,6,The central relationship of the novel is the passionate adulterous...
1,alanic_roses,hetero,deepseek/deepseek-v4-pro,R1,Estelle Gerfaux | Renaud Jonchère,female_male,dyadic,romantic_relationship,reciprocal,unclear,none,explicit,hidden_except,main,positive,separation_erasure,high,8,"Relation centrale du premier arc du roman, structurant la désillus..."
2,alanic_roses,hetero,deepseek/deepseek-v4-pro,R2,Estelle Gerfaux | Vincent Marcenat,female_male,dyadic,married,reciprocal,exclusive,none,explicit,explicit,main,positive,stable_or_open,high,8,"Relation centrale du roman, initialement présentée comme une union..."
3,alanic_roses,hetero,deepseek/deepseek-v4-pro,R3,Adrien Gerfaux | Monique Françon,female_male,dyadic,married,reciprocal,exclusive,none,explicit,explicit,secondary,positive,stable_or_open,high,3,"Relation approuvée par le curé et les familles, menant au mariage."
4,alanic_roses,hetero,deepseek/deepseek-v4-pro,R4,Vincent Marcenat | Odette de Tintaniac,female_male,dyadic,married,asymmetrical,adultery,unclear,coded,explicit,secondary,neutral_ambivalent,fatal_ending,high,4,"Relation décrite comme un 'mariage d'amour' déçu, ayant évolué ver..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
64,gautier_usurpateur,hetero,deepseek/deepseek-v4-pro,R1,Fidé-Yori | Omiti,female_male,dyadic,romantic_relationship,reciprocal,exclusive,none,explicit,hidden_except,main,positive,stable_or_open,high,10,"Relation amoureuse principale du roman, explicite et réciproque, q..."
65,gautier_usurpateur,hetero,deepseek/deepseek-v4-pro,R2,"Ivakoura Teroumoto Mori, prince de Nagato | La Kisaki",female_male,dyadic,romantic_relationship,reciprocal,unclear,external_adultery,explicit,hidden_except,main,decadent_sublime,fatal_ending,high,9,"Relation amoureuse réciproque mais impossible, car la Kisaki est l..."
66,gautier_usurpateur,hetero,deepseek/deepseek-v4-pro,R3,"Ivakoura Teroumoto Mori, prince de Nagato | Fatkoura",female_male,dyadic,engaged,one_sided,exclusive,loyal,explicit,explicit,secondary,neutral_ambivalent,fatal_ending,high,7,Relation marquée par des fiançailles imposées et un amour unilatér...
67,gautier_usurpateur,hetero,deepseek/deepseek-v4-pro,R4,Yodogimi | Harounaga,female_male,dyadic,romantic_relationship,reciprocal,unclear,none,explicit,explicit,secondary,moralized_pathologized,unclear,high,5,Relation adultère (du point de vue de la mémoire de Taïko-Sama) en...


## 3. Read the relations one by one

The wide table is hard to read across eleven fields; this prints each relation as a block.

In [ ]:
for _, row in relations.iterrows():
    print(f"=== {row['relation_id']}  {row['characters']}")
    print(f"    who    : {row['relation_type']} / {row['structure']}")
    print(f"    bond   : {row['status']} / {row['reciprocity']} / {row['exclusivity']} / {row['adultery']}")
    print(f"    shown  : {row['explicitness']}  (in plot: {row['explicitness_in_plot']})")
    print(f"    story  : {row['centrality']} / {row['framing']} / {row['outcome']}   [{row['confidence']}]")
    print(f"    comment: {row['comment']}")
    print()

if relations.empty:
    print("No relations in this run.")

## 4. Evidence table

One row per piece of evidence. `quote` is optional by design: it is filled when a specific
passage carries the label, and left empty when the evidence is a pattern across the novel.

In [ ]:
def evidence_rows(records):
    for r in records:
        annotation = r.get("annotation") or {}
        for rel in annotation.get("relations") or []:
            for i, ev in enumerate(rel.get("evidence") or [], start=1):
                yield {
                    "novel_id": r.get("novel_id"),
                    "relation_id": rel.get("relation_id"),
                    "evidence_index": i,
                    "quote": ev.get("quote"),
                    "locator": ev.get("locator"),
                    "description": ev.get("description"),
                }


evidence = pd.DataFrame(list(evidence_rows(records)), columns=EV_COLUMNS)
n_quotes = int(evidence["quote"].notna().sum())
print(f"{len(evidence)} evidence item(s); {n_quotes} carry a verbatim quote")
evidence

In [ ]:
evidence.loc[evidence["novel_id"]=="gautier_cruautesamour"]

## 5. Are the quotes real?

Every `quote` should be findable verbatim in the source text. Anything that is not is a
fabricated quotation. Matching ignores whitespace, case, curly apostrophes, and guillemets.

In [ ]:
def normalise(text):
    text = unicodedata.normalize("NFC", str(text))
    for a, b in (("\u2019", "'"), ("\u2018", "'"), ("\u00ab", ""), ("\u00bb", ""),
                 ("\u201c", '"'), ("\u201d", '"'), ("\u2014", "-"), ("\u2013", "-")):
        text = text.replace(a, b)
    return re.sub(r"\s+", " ", text).strip().lower()


source_chunks = {}

if CORPUS_PATH.exists():
    for line in CORPUS_PATH.read_text(encoding="utf-8").splitlines():
        if line.strip():
            rec = json.loads(line)
            source_chunks.setdefault(rec["novel_id"], []).append(rec["text"])

sources = {
    novel_id: normalise("\n".join(chunks))
    for novel_id, chunks in source_chunks.items()
}

evidence["quote_found"] = pd.NA
if sources and not evidence.empty:
    has_quote = evidence["quote"].notna()
    evidence.loc[has_quote, "quote_found"] = evidence.loc[has_quote].apply(
        lambda row: normalise(row["quote"]) in sources.get(row["novel_id"], ""), axis=1
    )
    checked = int(has_quote.sum())
    found = int((evidence["quote_found"] == True).sum())  # noqa: E712
    rate = 1 - found / checked if checked else 0.0
    print(f"{found}/{checked} quotes located verbatim  |  hallucination rate: {rate:.1%}")
    missing = evidence.loc[evidence["quote_found"] == False]  # noqa: E712
    if len(missing):
        print("\nNot found in the source text:")
        display(missing[["relation_id", "quote", "locator"]])
elif not sources:
    print(f"Source text not available at {CORPUS_PATH}; skipping the check.")

In [ ]:
def novel_key(x):
    return str(x).strip().casefold()


def normalise(text):
    text = unicodedata.normalize("NFKC", str(text))
    text = text.replace("\u00ad", "").replace("\ufeff", "")

    # Recolle les mots coupés par un retour à la ligne dans le texte source
    text = re.sub(r"(?<=\w)-\s*\n\s*(?=\w)", "", text)

    for a, b in (
        ("\u2019", "'"), ("\u2018", "'"),
        ("\u00ab", '"'), ("\u00bb", '"'),
        ("\u201c", '"'), ("\u201d", '"'),
        ("\u2014", "-"), ("\u2013", "-"),
        ("\u2026", "..."),
        ("\u00a0", " "),
    ):
        text = text.replace(a, b)

    return re.sub(r"\s+", " ", text).strip().casefold()


def normalise_loose(text):
    text = normalise(text)

    # Deuxième test : on ignore la ponctuation,
    # mais on conserve exactement la suite de mots.
    text = re.sub(r"[^\w\s]", " ", text, flags=re.UNICODE)

    return re.sub(r"\s+", " ", text).strip()


# --------------------------------------------------
# 1. Charger TOUS les chunks
# --------------------------------------------------

def novel_key(x):
    """
    Normalise l'identifiant du roman et retire
    les suffixes de partie comme __p1, __p2, etc.
    """
    key = str(x).strip().casefold()
    key = re.sub(r"__p\d+$", "", key)
    return key


source_chunks = {}

if CORPUS_PATH.exists():
    for line in CORPUS_PATH.read_text(
        encoding="utf-8-sig"
    ).splitlines():

        if not line.strip():
            continue

        rec = json.loads(line)

        # andr_gide_faux_monnayeurs__p1
        # devient
        # andr_gide_faux_monnayeurs
        key = novel_key(rec.get("novel_id"))

        txt = rec.get("text")


        if key and txt:
            source_chunks.setdefault(key, []).append(str(txt))


# Reconstruction du texte complet du roman
sources = {
    key: normalise(" ".join(chunks))
    for key, chunks in source_chunks.items()
}

sources_loose = {
    key: normalise_loose(" ".join(chunks))
    for key, chunks in source_chunks.items()
}

# --------------------------------------------------
# 2. DIAGNOSTIC : vérifier que les IDs correspondent
# --------------------------------------------------

evidence_ids = {
    novel_key(x)
    for x in evidence["novel_id"].dropna().unique()
}

print("novel_id dans evidence :", sorted(evidence_ids))
print("novel_id dans corpus   :", sorted(sources.keys()))

for key in sorted(evidence_ids):
    print(
        f"{key!r}: "
        f"{len(sources.get(key, '')):,} caractères source"
    )


# --------------------------------------------------
# 3. Vérifier chaque citation
# --------------------------------------------------

def locate_quote(row):

    q = row.get("quote")

    if pd.isna(q) or not str(q).strip():
        return pd.NA

    key = novel_key(row.get("novel_id"))

    source = sources.get(key)

    if not source:
        return "NO_SOURCE"

    q_norm = normalise(q)

    # citation retrouvée après normalisation légère
    if q_norm in source:
        return "FOUND"

    # même texte mais ponctuation différente
    q_loose = normalise_loose(q)

    if q_loose and q_loose in sources_loose[key]:
        return "FOUND_PUNCT"

    return "NOT_FOUND"


evidence["quote_status"] = evidence.apply(
    locate_quote,
    axis=1
)

evidence["quote_found"] = evidence["quote_status"].isin(
    ["FOUND", "FOUND_PUNCT"]
)


# --------------------------------------------------
# 4. Résumé
# --------------------------------------------------

checked = evidence["quote_status"].notna()

found = evidence.loc[
    checked,
    "quote_found"
].sum()

print(
    f"\n{int(found)}/{int(checked.sum())} "
    "citations retrouvées"
)

print(
    evidence["quote_status"].value_counts(
        dropna=False
    )
)


# --------------------------------------------------
# 5. Afficher uniquement les problèmes
# --------------------------------------------------

missing = evidence.loc[
    evidence["quote_status"].isin(
        ["NO_SOURCE", "NOT_FOUND"]
    ),
    [
        "novel_id",
        "relation_id",
        "quote",
        "locator",
        "quote_status",
    ]
]

if not missing.empty:
    print("\nCitations à examiner :")
    display(missing)

## 6. Label distribution

How often each label was used, per closed field. Useful for spotting a field the model
collapsed onto a single value.

In [ ]:
dft = relations.groupby("relation_type").explicitness_in_plot.value_counts().unstack()
dft
#dft.divide(dft.sum(axis=1),axis=0)

In [ ]:
counts = []
for field in CLOSED_FIELDS:
    for label, n in Counter(relations[field].dropna()).most_common():
        counts.append({"field": field, "label": label, "n": n})

label_counts = pd.DataFrame(counts, columns=["field", "label", "n"])
label_counts

## 7. Write the Excel workbook

Four sheets: `Relations`, `Evidence`, `Label_counts`, and `Run_metadata` (model, timestamps,
and the SHA-256 hashes of the schema, codebook, and prompts used for the run).

In [ ]:
META_FIELDS = [
    "novel_id", "title", "author", "year", "model", "ok",
    "started_at_utc", "finished_at_utc",
    "input_num_chars", "truncated", "input_sha256",
    "schema_path", "schema_sha256", "codebook_path", "codebook_sha256",
    "system_prompt_sha256", "user_template_sha256",
]

meta_rows = []
for r in records:
    row = {f: r.get(f) for f in META_FIELDS}
    row["error"] = r.get("runtime_error") or r.get("parse_error") or r.get("validation_error")
    row["n_relations"] = len((r.get("annotation") or {}).get("relations") or [])
    meta_rows.append(row)

run_metadata = pd.DataFrame(meta_rows, columns=META_FIELDS + ["error", "n_relations"])

XLSX_PATH.parent.mkdir(parents=True, exist_ok=True)
with pd.ExcelWriter(XLSX_PATH, engine="openpyxl") as writer:
    relations.to_excel(writer, sheet_name="Relations", index=False)
    evidence.to_excel(writer, sheet_name="Evidence", index=False)
    label_counts.to_excel(writer, sheet_name="Label_counts", index=False)
    run_metadata.to_excel(writer, sheet_name="Run_metadata", index=False)

print(f"Wrote {XLSX_PATH}")
print(f"  Relations    {len(relations)} rows")
print(f"  Evidence     {len(evidence)} rows")
print(f"  Label_counts {len(label_counts)} rows")
print(f"  Run_metadata {len(run_metadata)} rows")